In [0]:
# ============================================================
# CELDA 1 — Importaciones y carga de datos
# ============================================================
from pyspark.ml.feature import VectorAssembler, StringIndexer, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql.functions import when, col

df = spark.table("wine_quality_clean")

# ============================================================
# CELDA 2 — Preparar etiqueta binaria (1 = calidad alta, 0 = no)
# ============================================================
df_ml = df.withColumn("label", when(col("quality") >= 7, 1.0).otherwise(0.0))

# Verificar balance de clases
display(df_ml.groupBy("label").count())

# ============================================================
# CELDA 3 — Codificar variable categórica wine_type
# ============================================================
indexer = StringIndexer(inputCol="wine_type", outputCol="wine_type_index")

# Variables numéricas de entrada
feature_cols = [
    "fixed_acidity", "volatile_acidity", "citric_acid", "residual_sugar",
    "chlorides", "free_sulfur_dioxide", "total_sulfur_dioxide",
    "density", "pH", "sulphates", "alcohol", "wine_type_index"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
scaler    = StandardScaler(inputCol="features", outputCol="scaled_features")

# ============================================================
# CELDA 4 — Definir y entrenar el modelo
# ============================================================
rf = RandomForestClassifier(
    featuresCol="scaled_features",
    labelCol="label",
    numTrees=100,
    maxDepth=8,
    seed=42
)

pipeline = Pipeline(stages=[indexer, assembler, scaler, rf])

# División 80% entrenamiento / 20% prueba
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

print(f"Datos de entrenamiento: {train_df.count()}")
print(f"Datos de prueba:        {test_df.count()}")

model = pipeline.fit(train_df)
predictions = model.transform(test_df)

# ============================================================
# CELDA 5 — Evaluar el modelo
# ============================================================
# AUC (Area Under ROC Curve)
evaluator_auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
auc = evaluator_auc.evaluate(predictions)

# Exactitud (Accuracy)
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")
acc = evaluator_acc.evaluate(predictions)

# F1-Score
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")
f1 = evaluator_f1.evaluate(predictions)

print(f"AUC-ROC:  {auc:.4f}")
print(f"Accuracy: {acc:.4f}")
print(f"F1-Score: {f1:.4f}")

# ============================================================
# CELDA 6 — Importancia de variables (¿qué más influye en la calidad?)
# ============================================================
import pandas as pd

rf_model = model.stages[-1]
feature_importances = pd.DataFrame({
    "feature":    feature_cols,
    "importance": rf_model.featureImportances.toArray()
}).sort_values("importance", ascending=False)

print(feature_importances.to_string(index=False))

# Convertir a Spark para visualizar
display(spark.createDataFrame(feature_importances))
# Visualizar como Bar Chart horizontal: X = importance, Y = feature

# ============================================================
# CELDA 7 — Guardar predicciones para el dashboard
# ============================================================
predictions.select("quality", "label", "prediction", "probability", "wine_type") \
    .write.format("delta").mode("overwrite").saveAsTable("wine_ml_predictions")

print("✅ Predicciones guardadas en 'wine_ml_predictions'")